In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import pyro
import pyro.distributions as dist
from pyro.nn import PyroModule, PyroSample
import torch.nn as nn
from pyro.infer import Predictive

###PCA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

###Forward and Backward Selection
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
import statsmodels.api as sm

# HMC
from pyro.infer import MCMC, NUTS

# variational inference
from pyro.infer import SVI, Trace_ELBO
from pyro.infer.autoguide import AutoDiagonalNormal, AutoMultivariateNormal
from tqdm.auto import trange

import matplotlib as mpl
import os
import sys
import math

: 

In [ ]:
sys.path.append(
    r"C:\Users\thumo\OneDrive - Georgia Institute of Technology\Georgia Tech\Semesters\Spring 2025\CSE 8803 IUQ\Project\2-BNN_trained_prior\you-need-a-good-prior"
)

: 

In [3]:
from optbnn.gp.models.gpr import GPR
from optbnn.gp import kernels, mean_functions, priors
from optbnn.bnn.reparam_nets import GaussianMLPReparameterization
from optbnn.bnn.nets.mlp import MLP
from optbnn.bnn.likelihoods import LikGaussian
from optbnn.bnn.priors import FixedGaussianPrior, OptimGaussianPrior
from optbnn.prior_mappers.wasserstein_mapper import MapperWasserstein, WassersteinDistance
from optbnn.utils.rand_generators import MeasureSetGenerator, GridGenerator
from optbnn.utils.normalization import normalize_data
from optbnn.utils.exp_utils import get_input_range
from optbnn.metrics.sampling import compute_rhat_regression
from optbnn.metrics import uncertainty as uncertainty_metrics
from optbnn.sgmcmc_bayes_net.regression_net import RegressionNet
from optbnn.utils import util

### 1. concatenated records

#### 1.1 records before GDPR

In [4]:
df_pre = pd.read_csv("pregdprApril2016_Vodafone.csv")
df_pre = df_pre.dropna()
df_pre.head()

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,target
0,2.651204e+10,2.1980,10.0,1.0,4.333936,2.651204e+10,2.2114,10.0,1.0,4.333936,...,2.2200,10.0,1.0,4.333936,2.651204e+10,2.2530,10.0,1.0,4.333936,2.256087
1,2.651204e+10,2.2114,10.0,1.0,4.333936,2.651204e+10,2.2240,10.0,1.0,4.333936,...,2.2530,10.0,1.0,4.333936,2.651204e+10,2.2760,10.0,1.0,4.333936,2.266113
2,2.651204e+10,2.2240,10.0,1.0,4.333936,2.651204e+10,2.2200,10.0,1.0,4.333936,...,2.2760,10.0,1.0,4.333936,2.651204e+10,2.2755,10.0,1.0,4.333936,2.240884
3,2.651204e+10,2.2200,10.0,1.0,4.333936,2.651204e+10,2.2530,10.0,1.0,4.333936,...,2.2755,10.0,1.0,4.333936,2.651204e+10,2.2750,10.0,1.0,4.333936,2.248944
4,2.651204e+10,2.2530,10.0,1.0,4.333936,2.651204e+10,2.2760,10.0,1.0,4.333936,...,2.2750,10.0,1.0,4.333936,2.651204e+10,2.2670,10.0,1.0,4.333936,2.266673


In [5]:
X_pre = df_pre.iloc[:, : -1].values
X_pre, X_pre.shape, type(X_pre)

(array([[2.65120381e+10, 2.19800000e+00, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 4.33393580e+00],
        [2.65120381e+10, 2.21140000e+00, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 4.33393580e+00],
        [2.65120381e+10, 2.22400000e+00, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 4.33393580e+00],
        ...,
        [2.65585703e+10, 2.28050000e+00, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 4.54759913e+00],
        [2.65585703e+10, 2.30600000e+00, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 4.54759913e+00],
        [2.65585703e+10, 2.30300000e+00, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 4.54759913e+00]]),
 (266, 25),
 numpy.ndarray)

In [6]:
y_pre = df_pre["target"].values
y_pre, y_pre.shape, type(y_pre)

(array([2.25608707, 2.26611326, 2.24088429, 2.24894415, 2.26667281,
        2.27921509, 2.25571762, 2.26159988, 2.29436801, 2.31076374,
        2.2981389 , 2.29259734, 2.33881947, 2.34882441, 2.34724857,
        2.34957422, 2.39757887, 2.39982373, 2.36698951, 2.37917466,
        2.39162163, 2.42290052, 2.40792925, 2.4111102 , 2.39836166,
        2.29655652, 2.41208678, 2.44052516, 2.55508424, 2.54739523,
        2.53155507, 2.56380438, 2.61593893, 2.60332905, 2.62804075,
        2.58984206, 2.58360664, 2.5353072 , 2.57237683, 2.55994945,
        2.55114918, 2.46733855, 2.4573934 , 2.45395336, 2.46761844,
        2.4400321 , 2.46642801, 2.46867143, 2.48560941, 2.4823461 ,
        2.47393251, 2.47880759, 2.45694342, 2.45956758, 2.45543156,
        2.47765469, 2.49604054, 2.45742518, 2.47829313, 2.44532202,
        2.48250516, 2.45494274, 2.47987089, 2.45319871, 2.4552506 ,
        2.42778713, 2.42860595, 2.42347459, 2.43713708, 2.42584995,
        2.40684763, 2.40891392, 2.49481048, 2.50

#### 1.2 records after GDPR

In [7]:
df_post = pd.read_csv("postgdprMay2018_Vodafone.csv")
df_post = df_post.dropna()
df_post.head()

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,target
0,2.667656e+10,1.949200,10.0,1.0,5.090377,2.667656e+10,1.951200,10.0,1.0,5.090377,...,1.875200,10.0,1.0,5.334961,2.667656e+10,1.857485,10.0,1.0,5.334961,2.708364
1,2.667656e+10,1.951200,10.0,1.0,5.090377,2.667656e+10,1.938000,10.0,1.0,5.090377,...,1.857485,10.0,1.0,5.334961,2.667656e+10,1.881400,10.0,1.0,5.334961,2.703459
2,2.667656e+10,1.938000,10.0,1.0,5.090377,2.667656e+10,1.875200,10.0,1.0,5.334961,...,1.881400,10.0,1.0,5.334961,2.667656e+10,1.879478,10.0,1.0,5.334961,2.659580
3,2.667656e+10,1.875200,10.0,1.0,5.334961,2.667656e+10,1.857485,10.0,1.0,5.334961,...,1.879478,10.0,1.0,5.334961,2.667656e+10,1.851400,10.0,1.0,5.334961,2.668940
4,2.667656e+10,1.857485,10.0,1.0,5.334961,2.667656e+10,1.881400,10.0,1.0,5.334961,...,1.851400,10.0,1.0,5.334961,2.667656e+10,1.849400,10.0,1.0,5.334961,2.643431


In [8]:
X_post = df_post.iloc[:, : -1].values
X_post, X_post.shape, type(X_post)

(array([[2.66765608e+10, 1.94920000e+00, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 5.33496083e+00],
        [2.66765608e+10, 1.95120000e+00, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 5.33496083e+00],
        [2.66765608e+10, 1.93800000e+00, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 5.33496083e+00],
        ...,
        [2.67708437e+10, 1.47220000e+00, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 5.78102990e+00],
        [2.67708437e+10, 1.47670000e+00, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 5.78102990e+00],
        [2.67708437e+10, 1.49520000e+00, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 5.78102990e+00]]),
 (395, 25),
 numpy.ndarray)

In [9]:
y_post = df_post["target"].values
y_post, y_post.shape, type(y_post)

(array([2.70836436, 2.70345883, 2.65957988, 2.66893988, 2.64343075,
        2.66469298, 2.68557857, 2.68567736, 2.67536531, 2.68810711,
        2.69083718, 2.66673213, 2.66877771, 2.67723624, 2.66703526,
        2.66450988, 2.69535577, 2.71765503, 2.7099193 , 2.75590914,
        2.73220125, 2.71121629, 2.65619064, 2.63088866, 2.61248312,
        2.61391539, 2.60011567, 2.58482005, 2.56272377, 2.55528832,
        2.56265476, 2.5419655 , 2.5106086 , 2.52029582, 2.55458132,
        2.65453642, 2.63991757, 2.62714552, 2.60359116, 2.64329502,
        2.66503516, 2.659148  , 2.6689257 , 2.65385365, 2.63721542,
        2.62378298, 2.5891709 , 2.56069197, 2.58350383, 2.56821423,
        2.57082953, 2.56226977, 2.55912685, 2.56742504, 2.55959258,
        2.55725735, 2.51897814, 2.44792163, 2.42158372, 2.44860749,
        2.43004275, 2.43042684, 2.46173302, 2.47957249, 2.4792912 ,
        2.4889612 , 2.48351234, 2.50991026, 2.49357423, 2.50346213,
        2.51702872, 2.50832706, 2.48358049, 2.51

### 2. BNN and plot functions

In [10]:
class BNN(PyroModule):
    def __init__(self, weight_prior, bias_prior, in_dim=1, out_dim=1, hid_dim=10, n_hid_layers=5):
        '''
        functional model (network architecture):
            a fully connected neural network.

        stochastic model:
            Gaussian prior on weight and bias: p(theta) ~ dist.Normal(0., weight_prior or bias_prior), where weight_prior and bias_prior are learned;
            Gaussian likelihood function: p(y | x, theta) ~ dist.Normal(functional model(x), sigma^2), where sigma ~ dist.Gamma(.5, 1).
        '''
        super().__init__()

        # make sure the dimensions are valid
        assert in_dim > 0 and out_dim > 0 and hid_dim > 0 and n_hid_layers > 0

        # activation function for the whole network, can also be ReLU or LeakyReLU
        self.activation = nn.Tanh()

        # define the layer sizes and the PyroModule layer list
        self.layer_sizes = [in_dim] + n_hid_layers * [hid_dim] + [out_dim]
        layer_list = [PyroModule[nn.Linear](self.layer_sizes[idx - 1], self.layer_sizes[idx]) for idx in
                      range(1, len(self.layer_sizes))]
        self.layers = PyroModule[torch.nn.ModuleList](layer_list)

        # set the probability distribution for each layer's weight and bias
        for layer_idx, layer in enumerate(self.layers):
            layer.weight = PyroSample(dist.Normal(0., weight_prior[layer_idx]).expand([self.layer_sizes[layer_idx + 1], self.layer_sizes[layer_idx]]).to_event(2))
            layer.bias = PyroSample(dist.Normal(0., bias_prior[layer_idx]).expand([self.layer_sizes[layer_idx + 1]]).to_event(1))

    def forward(self, x, y=None):
        # functional model(x)
        # input --> hidden
        x = self.activation(self.layers[0](x))
        # hidden --> hidden
        for layer in self.layers[1:-1]:
            x = self.activation(layer(x))
        # hidden --> output
        mu = self.layers[-1](x).squeeze()

        # sample from P(y | x, \theta)
        sigma = pyro.sample("sigma", dist.Gamma(.5, 1))
        with pyro.plate("data", x.shape[0]):
            # obs is used when quantifying and visualizing the uncertainty of predictions
            obs = pyro.sample("obs", dist.Normal(mu, sigma * sigma), obs=y)
        
        return mu

In [11]:
def plot_predictions(preds, y):
    '''
    Function to visualize the predictions and the uncertainty of predictions.
    '''
    y_pred = preds['obs'].T.detach().numpy().mean(axis=1)
    y_std = preds['obs'].T.detach().numpy().std(axis=1)

    fig, ax = plt.subplots(figsize=(10, 5))

    # decide the range of the y axis based on the number of the labels
    time_idx = np.array(range(len(y)))
    xlims = [time_idx.min() - 0.1, time_idx.max() + 0.1]
    # decide the range of the y axis based on the range of the labels
    ylims = [min(y.min(), y_pred.min()) - 20,
             max(y.max(), y_pred.max()) + 20]
    
    plt.xlim(xlims)
    plt.ylim(ylims)
    plt.xlabel("time", fontsize=20)
    plt.ylabel("closing price", fontsize=20)

    ax.plot(time_idx, y, 'ko', markersize=1, label="observations")
    ax.plot(time_idx, y_pred, '-', linewidth=0.5, color="#408765", label="predictive mean")
    ax.fill_between(time_idx, y_pred - 2 * y_std, y_pred + 2 * y_std, alpha=0.6, color='#86cfac', zorder=0)

    plt.legend(loc=4, fontsize=15, frameon=False)

In [12]:
def plot_uncertainty(preds, y):
    '''
    Function to visualize only the uncertainty.
    '''
    fig, ax = plt.subplots(figsize=(10, 5))

    time_idx = np.array(range(len(y)))
    y_std = preds['obs'].T.detach().numpy().std(axis=1)

    xlims = [time_idx.min() - 0.1, time_idx.max() + 0.1]
    ylims = [y_std.min() - 0.5, y_std.max() + 0.5]

    plt.xlim(xlims)
    plt.ylim(ylims)
    plt.xlabel("time", fontsize=20)
    plt.ylabel("std of closing price", fontsize=20)

    ax.plot(time_idx, y_std, 'ko', markersize=1, label="std of predictions")
    ax.plot(time_idx, y_std, '-', linewidth=0.5, color="#408765")

    plt.legend(loc=4, fontsize=15, frameon=False)

### 3. uncertainty quantification of data pre GDPR

#### 3.1 Train the prior of BNN.

In [13]:
###IGNORE THIS IF YOU DON'T HAVE CUDA
import torch

print("CUDA Available:", torch.cuda.is_available())
print("CUDA Device Count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("CUDA Device Name:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")
print(torch.version.cuda)
print(torch.cuda.device_count())

CUDA Available: True
CUDA Device Count: 1
CUDA Device Name: NVIDIA GeForce RTX 4070
Training on device: cuda
11.8
1


In [14]:
%time 
noise_var = 0.1
n_units = 128
n_hidden = 1
activation_fn = "tanh"
num_iters = 200  # Number of iteterations of Wasserstein optimization
lr = 0.05        # The learning rate
n_samples = 128  # The mini-batch size
out_dir = "./exp/gdpr/optim_gaussian"

X_pre_n, y_pre_n, y_mean, y_std = normalize_data(X_pre, y_pre)
x_min, x_max = get_input_range(X_pre_n, X_pre_n)
epsilon = 1e-6
x_min = np.minimum(x_min, x_max - epsilon)
input_dim, output_dim = int(X_pre.shape[-1]), 1
    
# Initialize the measurement set generator
rand_generator = MeasureSetGenerator(X_pre_n, x_min, x_max, 0.7)

# Initialize the mean and covariance function of the target hierarchical GP prior
mean = mean_functions.Zero()
    
lengthscale = math.sqrt(2. * input_dim)
variance = 1.
kernel = kernels.RBF(input_dim=input_dim,
                     lengthscales=torch.tensor([lengthscale], dtype=torch.double),
                     variance=torch.tensor([variance], dtype=torch.double), ARD=True)

# Place hyper-priors on lengthscales and variances
kernel.lengthscales.prior = priors.LogNormal(
    torch.ones([input_dim]) * math.log(lengthscale),
    torch.ones([input_dim]) * 1.)
kernel.variance.prior = priors.LogNormal(
    torch.ones([1]) * 0.1,
    torch.ones([1]) * 1.)
        
# Initialize the GP model
gp = GPR(X=torch.from_numpy(X_pre_n), Y=torch.from_numpy(y_pre_n).reshape([-1, 1]),
             kern=kernel, mean_function=mean)
gp.likelihood.variance.set(noise_var)
    
# Initialize tunable MLP prior
hidden_dims = [n_units] * n_hidden
mlp_reparam = GaussianMLPReparameterization(input_dim, output_dim,
    hidden_dims, activation_fn, scaled_variance=True)
    
mapper = MapperWasserstein(gp, mlp_reparam, rand_generator, out_dir=out_dir,
                               output_dim=output_dim, n_data=100,
                               wasserstein_steps=(0, 200), ###more than 200
                               wasserstein_lr=0.02,
                               logger=None, wasserstein_thres=0.1,
                               n_gpu=1, gpu_gp=True) ##Change GPU if you don't have CUDA; same thing for the post training
    
w_hist = mapper.optimize(num_iters=num_iters, n_samples=n_samples,
                             lr=lr, print_every=10, save_ckpt_every=10, debug=True)

print("----" * 20)

CPU times: total: 0 ns
Wall time: 0 ns


C:\Users\thumo\OneDrive - Georgia Institute of Technology\Georgia Tech\Semesters\Spring 2025\CSE 8803 IUQ\Project\2-BNN_trained_prior\you-need-a-good-prior\optbnn\gp\parameter.py:63: UserWarning: An output with one or more elements was resized since it had shape [1], which does not match the required output shape []. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\Resize.cpp:37.)
  return torch.log(torch.exp(t) - 1, out=out)
C:\Users\thumo\OneDrive - Georgia Institute of Technology\Georgia Tech\Semesters\Spring 2025\CSE 8803 IUQ\Project\2-BNN_trained_prior\you-need-a-good-prior\optbnn\gp\models\model.py:136: UserWarning: torch.cholesky is deprecated in favor of torch.linalg.cholesky and will be removed in a future PyTorch

>>> Iteration #   1: Wasserstein Dist 8.4359
>>> Iteration #  10: Wasserstein Dist 2.1295
>>> Iteration #  20: Wasserstein Dist 1.7561
>>> Iteration #  30: Wasserstein Dist 0.8382
>>> Iteration #  40: Wasserstein Dist 1.0130
>>> Iteration #  50: Wasserstein Dist 2.3907
>>> Iteration #  60: Wasserstein Dist 1.3488
>>> Iteration #  70: Wasserstein Dist 1.2061
>>> Iteration #  80: Wasserstein Dist 0.7444
>>> Iteration #  90: Wasserstein Dist 1.0208
>>> Iteration # 100: Wasserstein Dist 0.6187
>>> Iteration # 110: Wasserstein Dist 2.0156
>>> Iteration # 120: Wasserstein Dist 0.6532
>>> Iteration # 130: Wasserstein Dist -0.3169
>>> Iteration # 140: Wasserstein Dist -0.4783
>>> Iteration # 150: Wasserstein Dist 1.4382
>>> Iteration # 160: Wasserstein Dist 0.3840
>>> Iteration # 170: Wasserstein Dist 1.0497
>>> Iteration # 180: Wasserstein Dist -0.7925
>>> Iteration # 190: Wasserstein Dist 2.4293
>>> Iteration # 200: Wasserstein Dist 0.1246
Saved intermediate wasserstein values in: ./exp/gdpr

In [15]:
for name, param in mlp_reparam.named_parameters():
    print(f"parameter name: {name}, parameter shape: {param}")

parameter name: layers.0.W_std, parameter shape: Parameter containing:
tensor([3.2722], device='cuda:0', requires_grad=True)
parameter name: layers.0.b_std, parameter shape: Parameter containing:
tensor([2.5010], device='cuda:0', requires_grad=True)
parameter name: output_layer.W_std, parameter shape: Parameter containing:
tensor([1.0945], device='cuda:0', requires_grad=True)
parameter name: output_layer.b_std, parameter shape: Parameter containing:
tensor([-0.6776], device='cuda:0', requires_grad=True)


In [16]:
def maintain_positivity(x):
    '''
    maintain the positivity of weight and bias standard derivations
    '''
    return np.log(1 + np.exp(x))

pre_weight_prior = [maintain_positivity(3.6905), maintain_positivity(1.3724)]
pre_bias_prior = [maintain_positivity(2.9497), maintain_positivity(-0.8432)]

#### 3.2 train the BNN

In [17]:
# clear parameters to ensure every training start from scratch
pyro.clear_param_store()

# set up BNN
model_VI = BNN(pre_weight_prior, pre_bias_prior, in_dim=25, out_dim=1, hid_dim=128, n_hid_layers=1)

#mean_field_guide = AutoDiagonalNormal(model_VI) # mean field variational inference
guide = AutoMultivariateNormal(model_VI) # use multivariate normal with full covariance to approxiamte posterior

# apply SGD to maximizing the ELBO
optimizer = pyro.optim.Adam({"lr": 0.001})
svi = SVI(model_VI, guide, optimizer, loss=Trace_ELBO())

# # clear parameters to avoid influencing others
pyro.clear_param_store()

In [18]:
###Better Rendering
from tqdm.notebook import trange
# or use tqdm.auto to auto-detect
from tqdm.auto import trange

In [ ]:
%time
num_epochs = 8000 # number of training epoches, 10000 now for quick test: 25000
progress_bar = trange(num_epochs) # show progress bar (only for visualization purpose)

X_pre_n_tensor = torch.tensor(X_pre_n, dtype=torch.float)
y_pre_n_tensor = torch.tensor(y_pre_n, dtype=torch.float)

for epoch in progress_bar:
    loss = svi.step(X_pre_n_tensor, y_pre_n_tensor)
    progress_bar.set_postfix(loss=f"{loss / X_pre_n.shape[0]:.3f}")
    if epoch % 500 == 0:
        print("[iteration %04d] loss: %.3f" % (epoch + 1, loss / X_pre_n.shape[0]))

CPU times: total: 0 ns
Wall time: 0 ns


  0%|          | 0/8000 [00:00<?, ?it/s]

[iteration 0001] loss: 3637.367
[iteration 0501] loss: 4443.420
[iteration 1001] loss: 922.404
[iteration 1501] loss: 684.091
[iteration 2001] loss: 78.889
[iteration 2501] loss: 95.459
[iteration 3001] loss: 215.201
[iteration 3501] loss: 124.708
[iteration 4001] loss: 588.566
[iteration 4501] loss: 161.079
[iteration 5001] loss: 53.195
[iteration 5501] loss: 54.718
[iteration 6001] loss: 109.448
[iteration 6501] loss: 68.942


In [ ]:
predictive = Predictive(model_VI, guide=guide, num_samples=1000)

preds = predictive(X_pre_n_tensor)

plot_predictions(preds, y_pre_n_tensor)

In [ ]:
pred_samples = preds["obs"]
pred_mean = pred_samples.mean(dim=0) 
# Calculate RMSE
y_true = y_pre_n_tensor
rmse = torch.sqrt(torch.mean((pred_mean - y_true) ** 2))
print(rmse)

In [ ]:
plot_uncertainty(preds, y_pre_n)

### 4. uncertainty quantification of data post GDPR

#### 4.1 Train the prior of BNN.

In [ ]:
noise_var = 0.1
n_units = 128
n_hidden = 1
activation_fn = "tanh"
num_iters = 250  # Number of iteterations of Wasserstein optimization
lr = 0.05        # The learning rate
n_samples = 128  # The mini-batch size
out_dir = "./exp/gdpr/optim_gaussian"

X_post_n, y_post_n, y_mean, y_std = normalize_data(X_post, y_post)
x_min, x_max = get_input_range(X_post_n, X_post_n)
epsilon = 1e-6
x_min = np.minimum(x_min, x_max - epsilon)
input_dim, output_dim = int(X_post.shape[-1]), 1
    
# Initialize the measurement set generator
rand_generator = MeasureSetGenerator(X_post_n, x_min, x_max, 0.7)
    
# Initialize the mean and covariance function of the target hierarchical GP prior
mean = mean_functions.Zero()
    
lengthscale = math.sqrt(2. * input_dim)
variance = 1.
kernel = kernels.RBF(input_dim=input_dim,
                     lengthscales=torch.tensor([lengthscale], dtype=torch.double),
                     variance=torch.tensor([variance], dtype=torch.double), ARD=True)

# Place hyper-priors on lengthscales and variances
kernel.lengthscales.prior = priors.LogNormal(
    torch.ones([input_dim]) * math.log(lengthscale),
    torch.ones([input_dim]) * 1.)
kernel.variance.prior = priors.LogNormal(
    torch.ones([1]) * 0.1,
    torch.ones([1]) * 1.)
        
# Initialize the GP model
gp = GPR(X=torch.from_numpy(X_post_n), Y=torch.from_numpy(y_post_n).reshape([-1, 1]),
             kern=kernel, mean_function=mean)
gp.likelihood.variance.set(noise_var)
    
# Initialize tunable MLP prior
hidden_dims = [n_units] * n_hidden
mlp_reparam = GaussianMLPReparameterization(input_dim, output_dim,
    hidden_dims, activation_fn, scaled_variance=True)
    
mapper = MapperWasserstein(gp, mlp_reparam, rand_generator, out_dir=out_dir,
                               output_dim=output_dim, n_data=100,
                               wasserstein_steps=(0, 250), ##Should be more than 200
                               wasserstein_lr=0.02,
                               logger=None, wasserstein_thres=0.1,
                               n_gpu=1, gpu_gp=True) ##Change GPU if you don't have CUDA; same thing for the PRE training
    
w_hist = mapper.optimize(num_iters=num_iters, n_samples=n_samples,
                             lr=lr, print_every=10, save_ckpt_every=10, debug=True)

print("----" * 20)

In [ ]:
for name, param in mlp_reparam.named_parameters():
    print(f"parameter name: {name}, parameter shape: {param}")

In [ ]:
def maintain_positivity(x):
    '''
    maintain the positivity of weight and bias standard derivations
    '''
    return np.log(1 + np.exp(x))

post_weight_prior = [maintain_positivity(3.1053), maintain_positivity(1.1434)]
post_bias_prior = [maintain_positivity(2.0997), maintain_positivity(-0.6430)]

#### 4.2 train the BNN

In [ ]:
# clear parameters to ensure every training start from scratch
pyro.clear_param_store()

# set up BNN
model_VI = BNN(post_weight_prior, post_bias_prior, in_dim=25, out_dim=1, hid_dim=128, n_hid_layers=1)

#mean_field_guide = AutoDiagonalNormal(model_VI) # mean field variational inference
guide = AutoMultivariateNormal(model_VI) # use multivariate normal with full covariance to approxiamte posterior

# apply SGD to maximizing the ELBO
optimizer = pyro.optim.Adam({"lr": 0.001})
svi = SVI(model_VI, guide, optimizer, loss=Trace_ELBO())

# # clear parameters to avoid influencing others
pyro.clear_param_store()

In [ ]:
num_epochs = 10000 # number of training epoches, 10000 now for quick test
progress_bar = trange(num_epochs) # show progress bar (only for visualization purpose)

X_post_n_tensor = torch.tensor(X_post_n, dtype=torch.float)
y_post_n_tensor = torch.tensor(y_post_n, dtype=torch.float)

for epoch in progress_bar:
    loss = svi.step(X_post_n_tensor, y_post_n_tensor)
    progress_bar.set_postfix(loss=f"{loss / X_post_n.shape[0]:.3f}")
    if epoch % 500 == 0:
        print("[iteration %04d] loss: %.3f" % (epoch + 1, loss / X_post_n.shape[0]))

In [ ]:
predictive = Predictive(model_VI, guide=guide, num_samples=1000)

preds = predictive(X_post_n_tensor)

plot_predictions(preds, y_post_n_tensor)

In [ ]:
plot_uncertainty(preds, y_post_n)

In [ ]:
##RMSE
pred_samples = preds["obs"]
pred_mean = pred_samples.mean(dim=0) 
# Calculate RMSE
y_true = y_post_n_tensor
rmse = torch.sqrt(torch.mean((pred_mean - y_true) ** 2))
print(rmse)